# Problem 4: Arabic Information Retrieval + RAG (Step-by-step)

We will build **three layers**: **classical search**, **semantic search**, and **RAG**.

## Analogy (simple)
- **Classical search (TF-IDF/BM25):** like a library index. It finds pages that contain the same keywords.
- **Semantic search:** like a librarian who understands meaning, even if the exact words are different.
- **RAG:** like asking a smart assistant who first reads the relevant pages, then answers in their own words.

## Requirements
- Use one small open-source Hugging Face LLM (1B-3B).
- We use **Qwen2.5-1.5B-Instruct**.
- We use a **single Arabic book** in plain text.
- Build classical + semantic retrieval, then compare RAG vs. LLM-only.

In [ ]:
# If needed, install dependencies (run once)
# !pip install sentence-transformers faiss-cpu transformers accelerate pandas scikit-learn

In [ ]:
import re
import time
import json
from pathlib import Path
import numpy as np
import pandas as pd
import faiss
from sklearn.feature_extraction.text import TfidfVectorizer
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE

## 1) Book selection and download
We pick a public-domain Arabic book. You can change the URL if needed.

Chosen book: **Al-Quran (simple text)**
Source (public): GitHub mirror.

In [ ]:
BOOK_URL = "https://raw.githubusercontent.com/quran/quran.com-frontend-v2/master/static/quran-simple.txt"
BOOK_TITLE = "Al-Quran (simple text)"
BOOK_PATH = Path("arabic_book.txt")

if not BOOK_PATH.exists():
    import urllib.request
    urllib.request.urlretrieve(BOOK_URL, BOOK_PATH)

text = BOOK_PATH.read_text(encoding="utf-8")
len(text)

## 2) Preprocessing + Chunking (2-4 sentences)
We split by Arabic punctuation. If not enough punctuation exists, we fallback to line-based chunks.

In [ ]:
def split_sentences_ar(text):
    # Split on Arabic and Latin punctuation
    parts = re.split(r'[\.\!\?؟]+', text)
    sentences = [p.strip() for p in parts if p.strip()]
    return sentences

def chunk_sentences(sentences, min_sents=2, max_sents=4):
    chunks = []
    i = 0
    while i < len(sentences):
        size = np.random.randint(min_sents, max_sents + 1)
        chunk = ". ".join(sentences[i:i+size])
        if chunk:
            chunks.append(chunk)
        i += size
    return chunks

sentences = split_sentences_ar(text)
if len(sentences) < 50:
    # fallback: split by lines if punctuation is limited
    lines = [l.strip() for l in text.splitlines() if l.strip()]
    sentences = lines

chunks = chunk_sentences(sentences, 2, 4)
len(chunks), chunks[:3]

## 3) Embeddings + FAISS index
We use a multilingual sentence embedding model that works well for Arabic.

In [ ]:
EMBED_MODEL = "intfloat/multilingual-e5-small"
embedder = SentenceTransformer(EMBED_MODEL)

# E5 expects a prefix such as 'query:' or 'passage:'
passages = ["passage: " + c for c in chunks]
embeddings = embedder.encode(passages, batch_size=32, show_progress_bar=True, normalize_embeddings=True)

dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(embeddings)

index.ntotal

## 4) Classical search (TF-IDF)
We build a TF-IDF index to retrieve top 5 chunks.

In [ ]:
tfidf = TfidfVectorizer(max_features=50000)
tfidf_matrix = tfidf.fit_transform(chunks)

def classical_search(query, top_k=5):
    q_vec = tfidf.transform([query])
    scores = (tfidf_matrix @ q_vec.T).toarray().squeeze()
    top_idx = scores.argsort()[::-1][:top_k]
    return [(chunks[i], float(scores[i])) for i in top_idx]

def semantic_search(query, top_k=5):
    q_emb = embedder.encode(["query: " + query], normalize_embeddings=True)
    scores, idx = index.search(q_emb, top_k)
    results = []
    for i, s in zip(idx[0], scores[0]):
        results.append((chunks[i], float(s)))
    return results

def compare_search(query):
    c = classical_search(query)
    s = semantic_search(query)
    rows = []
    for i in range(5):
        rows.append({
            "Rank": i + 1,
            "Classical (TF-IDF)": c[i][0],
            "C_Score": round(c[i][1], 4),
            "Semantic (Embeddings)": s[i][0],
            "S_Score": round(s[i][1], 4)
        })
    return pd.DataFrame(rows)

### Example query
Try with Arabic queries.

In [ ]:
compare_search("ما هي صفات المؤمنين؟")

## 5) RAG (LLM + retrieved context)
We will generate two answers:
- **RAG**: LLM sees top semantic chunks.
- **LLM-only**: no retrieved context.

We use Qwen2.5-1.5B-Instruct. If CPU is slow, reduce `max_new_tokens`.

In [ ]:
LLM_ID = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(LLM_ID)
model = AutoModelForCausalLM.from_pretrained(LLM_ID)
model = model.to(DEVICE)

def generate_answer(prompt, max_new_tokens=200):
    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=True, temperature=0.7)
    return tokenizer.decode(output[0], skip_special_tokens=True)

def answer_question(query):
    # Retrieve context
    top_chunks = semantic_search(query, top_k=3)
    context = "
".join([c[0] for c in top_chunks])

    rag_prompt = (
        "أنت مساعد يجيب بالعربية اعتمادا على السياق التالي فقط.
"
        "السياق:
" + context + "
"
        "السؤال: " + query + "
"
        "الإجابة: "
    )

    llm_prompt = (
        "أنت مساعد يجيب بالعربية.
"
        "السؤال: " + query + "
"
        "الإجابة: "
    )

    rag_answer = generate_answer(rag_prompt)
    llm_answer = generate_answer(llm_prompt)
    return rag_answer, llm_answer

## 6) Run 10 queries (required)
Create a list of 10 Arabic queries (mixed difficulty). Then run the comparison.

In [ ]:
queries = [
    "ما معنى الصبر؟",
    "اذكر آيات تتحدث عن الرحمة",
    "ما هي صفات المؤمنين؟",
    "ما قصة موسى؟",
    "ما هو وصف يوم القيامة؟",
    "ما علاقة التوبة بالمغفرة؟",
    "اذكر أمثلة على الشكر",
    "كيف يتحدث النص عن الصدق؟",
    "ما الفرق بين الإيمان والعمل؟",
    "ما هي عاقبة الظلم؟"
]

results = []
for q in queries:
    rag_ans, llm_ans = answer_question(q)
    results.append({
        "Query": q,
        "RAG_Answer": rag_ans,
        "LLM_Only_Answer": llm_ans
    })

pd.DataFrame(results)

# Required Deliverables (what to write in your report)
1. **Book description**: title, source URL, preprocessing, chunking, embedding, indexing.
2. **Search results**: at least 10 queries with classical + semantic top-5.
3. **RAG vs LLM-only**: 10 queries, compare outputs.
4. **Reflection**: which retrieval is more relevant? does RAG help? what improvements?

## Insights and suggestions
- Try a different embedding model (Arabic-specific).
- Try BM25 (e.g., `rank_bm25`).
- Try a different small LLM (e.g., 2B Arabic-friendly).
- Tune chunk size (1-3 vs. 3-5 sentences).